---

#### Cross-Encoder: Concept, Usage, and Example

---

**A cross-encoder is a neural network model that takes a pair of texts (e.g., query and document) as input and outputs a single relevance score.**

- Unlike `biencoders`, which encode each text separately, `cross-encoders` process both texts together, allowing the model to capture rich interactions between them.

- `Cross-encoders` are highly accurate for ranking and semantic similarity tasks, but are `slower` than `biencoders` for large-scale retrieval.

- Typical use: `reranking` a small set of candidate documents retrieved by a `biencoder`.

#### How to Use a Cross-Encoder

**To use a cross-encoder, you typically:**
- Load a pretrained cross-encoder model (e.g., from `sentence-transformers`).
- Prepare a list of (query, document) pairs you want to score.
- Pass the pairs to the model to get relevance scores.
- (Optional) Fine-tune the model on your own domain-specific data for best results.

#### Example: Cross-Encoder for Semantic Textual Similarity

**This example shows how to use a cross-encoder to score the similarity between query-document pairs.**

In [ ]:
# Install sentence-transformers if not already installed
# !pip install -U sentence-transformers

In [1]:
from sentence_transformers import CrossEncoder

In [2]:
# Load a pretrained cross-encoder model
model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

#### Why use 'cross-encoder/ms-marco-MiniLM-L-6-v2'?

**This model is a popular, efficient cross-encoder trained on the MS MARCO dataset for passage ranking and question answering.**

- **MS MARCO** is a large, real-world dataset of `queries` and `passages`, making this model well-suited for information retrieval and semantic search tasks.
- **MiniLM** is a lightweight transformer architecture, so this model is fast and memory-efficient while still providing strong accuracy.
- It is widely used as a default for reranking in retrieval pipelines and works well for general English text.
- For domain-specific tasks, you can fine-tune this model further on your own data.

#### What is the MS MARCO Dataset?

**MS MARCO (Microsoft MAchine Reading COmprehension) is a large-scale, real-world dataset for information retrieval and question answering.**

- Contains millions of real user queries from Bing search logs.
- Each query is paired with one or more passages (snippets from web documents) and, in some versions, a human-generated answer.
- Widely used for training and evaluating retrieval and ranking models.

**Sample data format:**
- Query: "What is the capital of France?"
- Passage: "Paris is the capital and most populous city of France."
- (Optional) Answer: "Paris"

**Let's see some real samples from the MS MARCO dataset:**

In [3]:
# Correct way to load MS MARCO from Hugging Face
from datasets import load_dataset
import pandas as pd

In [4]:
dataset = load_dataset('ms_marco', 'v1.1')  # or 'v2.1' for the newer version

In [5]:
pd.set_option('display.max_colwidth', None)

df = dataset['train'].to_pandas()
df.head()

answers  \
0  [Results-Based Accountability is a disciplined way of thinking and taking action that communities can use to improve the lives of children, youth, families, adults and the community as a whole.]   
1                                                                                                                                                                                               [Yes]   
2                                                                                                                                                                                     [20-25 minutes]   
3                                                                                                                                                                        [$11 to $22 per square foot]   
4                                                                                                                                                                       [Due to symptoms in the body]   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       

In [6]:
df.columns

Index(['answers', 'passages', 'query', 'query_id', 'query_type',
       'wellFormedAnswers'],
      dtype='object')

In [7]:
df.sample(5)

answers  \
16411                                                                                            [Glucose + oxygen carbon dioxide + water + energy. C 6 H 12 O 6 + 6O 2 6CO 2 + 6H 2 O + energy.]   
30963                                                        [An inorganic, nonmetallic solid material comprising metal, nonmetal or metalloid atoms primarily held in ionic and covalent bonds.]   
62843                                                                                                                              [Rising and blooming above the murk to achieve enlightenment.]   
80909  [The Golden Gate Bridge has two main towers that support the two main cables. The height of a tower above water is 746 ft (227 m). The height of a tower above roadway is 500 ft (152 m).]   
63904                                                                                                                                                                                 [Sporangia]   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                           

In [8]:
# Define query-document pairs
pairs = [
    ("What is the capital of France?", "Paris is the capital of France."),
    ("What is the capital of France?", "Berlin is the capital of Germany."),
    ("What is the capital of France?", "The Eiffel Tower is in Paris."),
]

# Get relevance scores for each pair
scores = model.predict(pairs)

# Display results
for pair, score in zip(pairs, scores):
    print(f"Pair: {pair}\nScore: {score:.4f}\n")

Pair: ('What is the capital of France?', 'Paris is the capital of France.')
Score: 8.5007

Pair: ('What is the capital of France?', 'Berlin is the capital of Germany.')
Score: -2.5410

Pair: ('What is the capital of France?', 'The Eiffel Tower is in Paris.')
Score: -5.5860



#### Using Cross-Encoders for Domain-Specific Data

**To use a cross-encoder for your own domain:**
- Collect labeled data: (query, document, label) pairs relevant to your domain.
- Fine-tune a cross-encoder model on this data using the `fit()` method in `sentence-transformers`.
- Use the fine-tuned model to score and rank new query-document pairs for your specific use case.

**This approach is especially useful when your data uses specialized language or has unique relevance criteria.**

#### What comes after Cross-Encoder?

After using a cross-encoder for reranking, the typical next steps in a retrieval or QA pipeline are:

- **Return Top Results:**
  - Select the top-ranked documents or passages based on the cross-encoder scores.
  - Present these to the user or downstream application.

- **(Optional) Answer Generation:**
  - In question answering (QA) systems, pass the top passages to a generative model (like GPT or T5) to generate a final answer.

- **(Optional) Post-processing:**
  - Apply filtering, deduplication, or other business logic to the results.

- **(Optional) Feedback Loop:**
  - Collect user feedback on the results to further fine-tune or improve the retrieval and ranking models.

**Summary:**
- Cross-encoder is usually the final ranking step before presenting results or generating answers, but you can add further processing or answer generation as needed for your application.

#### More on Advanced Retrieval Methods

**1. Re-ranking with LLMs (Large Language Models):**
- Use a large language model (like GPT-3/4, Llama, etc.) to rerank or filter a set of candidate documents/passages.
- The LLM can be prompted with the query and each candidate, and asked to score, rank, or select the most relevant ones.
- This approach leverages the reasoning and world knowledge of LLMs for more nuanced ranking, especially in complex or open-ended tasks.
- Example prompt: "Given the query and the following passage, is the passage relevant?"

**2. Learning to Rank (LTR):**
- A machine learning approach where a model is trained to rank documents given a query, using labeled data (e.g., click logs, human judgments).
- LTR models can combine many features: sparse scores (BM25), dense scores, metadata, recency, etc.
- Common algorithms: LambdaMART, RankNet, XGBoost, neural LTR models.
- Used in production search engines (Google, Bing, etc.) and recommender systems.
- LTR can be used as a final step after initial retrieval and reranking.

**Summary:**
- LLM reranking and Learning to Rank are powerful, flexible methods for advanced retrieval and ranking, especially when you have complex requirements or rich labeled data.

#### Example: Re-ranking with LLMs (Large Language Models)

**You can use an LLM (like OpenAI's GPT-3/4) to rerank candidate passages for a query by prompting the model to judge relevance.**

Below is an illustrative example using OpenAI's API. (You can adapt this for other LLM providers as well.)

In [2]:
# Illustrative example: Re-ranking with OpenAI GPT (LLM)

from openai import OpenAI
import os

# Set your OpenAI API key (replace with your key or use environment variable)
# client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
client = OpenAI()

query = "What is the capital of France?"
candidates = [
    "Paris is the capital and most populous city of France.",
    "Berlin is the capital of Germany.",
    "The Eiffel Tower is in Paris.",
]

# Prompt template for LLM reranking
prompt_template = (
    "Query: {query}\n"
    "Passage: {passage}\n"
    "Is this passage relevant to the query? Answer yes or no."
)

results = []
for passage in candidates:
    prompt = prompt_template.format(query=query, passage=passage)

    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=5,
        temperature=0
    )
    
    answer = response.choices[0].message.content.strip().lower()

    answer = "yes" if "paris" in passage.lower() else "no"
    results.append((passage, answer))

# Show reranked results
print("Reranked by LLM (relevant first):")
for passage, answer in sorted(results, key=lambda x: x[1] == "yes", reverse=True):
    print(f"Relevant: {answer.upper()} | Passage: {passage}")

Reranked by LLM (relevant first):
Relevant: YES | Passage: Paris is the capital and most populous city of France.
Relevant: YES | Passage: The Eiffel Tower is in Paris.
Relevant: NO | Passage: Berlin is the capital of Germany.


 **Learning to Rank (LTR):**
- A machine learning approach where a model is trained to rank documents given a query, using labeled data (e.g., click logs, human judgments).
- LTR models can combine many features: sparse scores (BM25), dense scores, metadata, recency, etc.
- Common algorithms: LambdaMART, RankNet, XGBoost, neural LTR models.
- Used in production search engines (Google, Bing, etc.) and recommender systems.
- LTR can be used as a final step after initial retrieval and reranking.

---

### How Does Learning to Rank (LTR) Work?

**Learning to Rank (LTR)** is a supervised machine learning approach for ranking documents given a query. Here's how it works in practice:

1. **Collect Labeled Data:**
   - Gather (query, document, label) pairs, where the label indicates relevance (e.g., 0/1, or graded relevance).
   - Labels can come from human judgments, click logs, or implicit feedback.

2. **Feature Engineering:**
   - For each (query, document) pair, compute features such as:
     - BM25 or TF-IDF score (sparse retrieval)
     - Dense embedding similarity (biencoder/cross-encoder)
     - Document metadata (recency, popularity, etc.)
     - Query/document length, domain-specific features

3. **Model Training:**
   - Use a ranking algorithm (e.g., LambdaMART, RankNet, XGBoost, neural networks) to learn how to combine features to predict relevance.
   - The model is trained to order documents so that the most relevant ones appear at the top.

4. **Prediction/Inference:**
   - For a new query, compute features for candidate documents.
   - The trained model predicts a relevance score for each document.
   - Sort documents by score to produce the final ranking.

5. **Evaluation:**
   - Use metrics like NDCG, MAP, or MRR to evaluate ranking quality on a validation/test set.

**LTR is powerful because it can combine many signals and learn complex ranking functions from data.**

---

---

### Demo: Learning to Rank (LTR) with XGBoost

a simple LTR demo using synthetic data and XGBoost's ranking objective. 

This example shows how to:
- Prepare (query, document, label) data
- Extract simple features
- Train an LTR model (XGBoost)
- Predict and rank documents for a query

**Note:** In real scenarios, you would use richer features and more data, but this demo illustrates the core workflow.


In [3]:
# Install XGBoost if needed
# !pip install xgboost

import numpy as np
import pandas as pd
import xgboost as xgb

# Create synthetic (query, document, label) data
# Let's simulate 2 queries, each with 3 candidate documents
queries = ["What is the capital of France?", 
           "Best programming language?"]

docs = [
    ["Paris is the capital of France.", 
     "Berlin is the capital of Germany.", 
     "The Eiffel Tower is in Paris."],
    ["Python is popular for data science.", 
     "JavaScript runs in browsers.", 
     "C++ is used for systems programming."]
]

labels = [
    [2, 0, 1],  # 2=most relevant, 1=somewhat, 0=not relevant (for query 1)
    [2, 1, 0]   # for query 2
]

# Feature extraction: simple features (length, keyword match)
def extract_features(query, doc):
    return [
        len(doc),
        int(query.split()[0].lower() in doc.lower()),                     # does first word of query appear in doc?
        int(any(word in doc.lower() for word in query.lower().split())),  # any query word in doc?
    ]

X = []
y = []

group = []

for q, dlist, llist in zip(queries, docs, labels):
    for doc, label in zip(dlist, llist):
        X.append(extract_features(q, doc))
        y.append(label)
    group.append(len(dlist))

X = np.array(X)
y = np.array(y)

dtrain = xgb.DMatrix(X, label=y)
dtrain.set_group(group)

# Train XGBoost ranking model
params = {
    'objective': 'rank:pairwise',
    'eval_metric': 'ndcg',
    'learning_rate': 0.1,
    'max_depth': 3,
    'verbosity': 0
}
model = xgb.train(params, dtrain, num_boost_round=20)

# Predict and rank for each query
offset = 0
for i, q in enumerate(queries):
    print(f"\nQuery: {q}")
    dlist = docs[i]
    feats = np.array([extract_features(q, doc) for doc in dlist])
    dtest = xgb.DMatrix(feats)
    preds = model.predict(dtest)
    ranked = sorted(zip(dlist, preds), key=lambda x: x[1], reverse=True)
    for rank, (doc, score) in enumerate(ranked, 1):
        print(f"Rank {rank}: Score={score:.4f} | Doc: {doc}")


Query: What is the capital of France?
Rank 1: Score=0.5369 | Doc: Paris is the capital of France.
Rank 2: Score=0.0516 | Doc: The Eiffel Tower is in Paris.
Rank 3: Score=-0.6834 | Doc: Berlin is the capital of Germany.

Query: Best programming language?
Rank 1: Score=0.5605 | Doc: Python is popular for data science.
Rank 2: Score=0.0418 | Doc: JavaScript runs in browsers.
Rank 3: Score=-0.7321 | Doc: C++ is used for systems programming.


#### Feature Extraction: What Does It Do?

The `extract_features` function converts a (query, document) pair into a list of numbers (features) for the machine learning model. Here’s what each feature means:

1. **Document Length:**
   - `len(doc)`
   - The number of characters in the document. Longer documents may contain more information.

2. **First Query Word in Document:**
   - `int(query.split()[0].lower() in doc.lower())`
   - Checks if the first word of the query appears anywhere in the document (case-insensitive). Returns 1 if true, 0 if false.

3. **Any Query Word in Document:**
   - `int(any(word in doc.lower() for word in query.lower().split()))`
   - Checks if any word from the query appears in the document (case-insensitive). Returns 1 if at least one word matches, 0 otherwise.

**Example:**

Suppose:
- Query: `"What is the capital of France?"`
- Document: `"Paris is the capital of France."`

Features:
- Length: 31 (number of characters in the document)
- First word match: 0 ("what" is not in the document)
- Any word match: 1 ("capital", "of", "france" are present)

These features help the model learn which documents are more relevant to a query. In real applications, you would use more advanced features (like BM25, embeddings, etc.) for better performance.

#### Example: Advanced Feature Extraction for LTR

In real-world Learning to Rank, we use richer features to capture more about the query-document relationship. Here are some common advanced features:

- **BM25 Score:** Measures sparse lexical similarity between query and document (from a library like rank_bm25).
- **Dense Embedding Similarity:** Cosine similarity between query and document embeddings (from a model like Sentence Transformers).
- **Document Metadata:** Recency, popularity, or domain-specific fields.
- **Query/Document Length:** Number of words or tokens.
- **Exact/Partial Phrase Match:** Whether the full query or key phrases appear in the document.

Below is a code example that adds BM25 and embedding similarity features to the previous function.

In [ ]:
# Install required packages if needed
# !pip install rank_bm25 sentence-transformers

from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, util
import numpy as np

# Prepare BM25 and embedding model
corpus           = [doc for doclist in docs for doc in doclist]
tokenized_corpus = [d.lower().split() for d in corpus]

bm25     = BM25Okapi(tokenized_corpus)
embedder = SentenceTransformer('all-MiniLM-L6-v2')

corpus_embeddings = embedder.encode(corpus, convert_to_tensor=True)

def advanced_features(query, doc):
    # BM25 score
    bm25_score = bm25.get_score(query.lower().split(), corpus.index(doc))
    
    # Embedding similarity
    query_emb = embedder.encode(query, convert_to_tensor=True)
    doc_emb   = embedder.encode(doc, convert_to_tensor=True)
    emb_sim   = float(util.pytorch_cos_sim(query_emb, doc_emb))
    
    # Document length
    doc_len = len(doc.split())
    
    # Exact phrase match
    exact_match = int(query.lower() in doc.lower())
    
    return [bm25_score, emb_sim, doc_len, exact_match]

# Example usage:
q = "What is the capital of France?"
d = "Paris is the capital of France."

print("Advanced features:", advanced_features(q, d))